In [ ]:
# =========================
# HostelGrid++ Colab Setup
# =========================

from google.colab import drive
drive.mount("/content/drive")

import os, sys, subprocess

PROJECT_DIR = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus"
REPO_URL = "https://github.com/anshu-ai-arch/hostalgrid-plus-plus.git"

if not os.path.exists(PROJECT_DIR):
    os.makedirs("/content/drive/MyDrive/hostelgrid-work", exist_ok=True)
    %cd /content/drive/MyDrive/hostelgrid-work
    !git clone https://github.com/anshu-ai-arch/hostalgrid-plus-plus.git

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

# Install core packages every new runtime.
!pip install -q -U numpy pydantic fastapi uvicorn
!pip install -q -U datasets transformers accelerate peft trl bitsandbytes

# torchao often causes PEFT version conflicts in Colab.
!pip uninstall -y torchao > /dev/null 2>&1

# Confirm paths.
DATA_DIR = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data"
OUTPUT_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs"
PLOT_DIR = "/content/drive/MyDrive/hostelgrid-work/plots"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print("Ready.")
print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)
print("Plots:", PLOT_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/anshu-ai-arch/hostalgrid-plus-plus.git

In [ ]:
!git clone https://github.com/anshu-ai-arch/hostalgrid-plus-plus
%cd hostalgrid-plus-plus

In [ ]:
!pip install numpy matplotlib pydantic -q


In [ ]:
import sys
sys.path.insert(0, '/content/hostalgrid-plus-plus')

from env import HostelGridEnv, Action, Observation, Reward
env = HostelGridEnv(mode='medium')
obs = env.reset()
print(f'Rooms: {len(obs.rooms)}')
print(f'Mode: {obs.mode}')
print('Setup OK ✓')


In [ ]:
from training import train_q, train_dqn, evaluate_all
from graders import grade_all

# Train Q-Learning
q_agent, q_hist = train_q(mode='medium', n_episodes=500)

# Train DQN
dqn_agent, dqn_hist = train_dqn(mode='medium', n_episodes=500)

# Evaluate
results = evaluate_all(q_agent, dqn_agent, mode='medium')

# Grade
print('\n=== Q-Learning ===')
grade_all(q_agent)

print('\n=== DQN ===')
grade_all(dqn_agent)

In [ ]:
import os

print(os.path.exists("/content/plots"))
print(os.listdir("/content"))


In [ ]:
!git clone https://github.com/anshu-ai-arch/hostalgrid-plus-plus.git


In [ ]:
!pip install -r requirements.txt

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    if "requirements.txt" in files:
        print(os.path.join(root, "requirements.txt"))


In [ ]:
%cd /content/hostalgrid-plus-plus
!pip install -r requirements.txt


In [ ]:
!pip install -q datasets transformers accelerate peft trl bitsandbytes



In [ ]:

from env.openenv_api import HostelGridOpenEnv, Action

env = HostelGridOpenEnv(task_id="task_medium")
obs = env.reset()
print(obs.model_dump())

obs, reward, done, info = env.step(Action(action_id=4))
print("reward:", reward.value, "score:", env.score(), "done:", done)
print(info)


In [ ]:
%%writefile /content/hostalgrid-plus-plus/env/openenv_api.py
from __future__ import annotations

from typing import Any, Dict, List
from pydantic import BaseModel

from env.hostelgrid_env import HostelGridEnv
from env.action import Action as RoomAction


class Action(BaseModel):
    action_id: int | None = None
    room_actions: List[int] | None = None


class OpenEnvObservation(BaseModel):
    power_usage: float
    avg_temperature: float = 27.0
    avg_occupancy: float
    complaint_level: int
    time_of_day: int
    carbon_rate: float
    current_cost: float
    rooms: List[Dict[str, Any]]
    mode: str
    heatwave: bool


class OpenEnvReward(BaseModel):
    value: float
    normalized: float


class HostelGridOpenEnv:
    def __init__(self, task_id: str = "task_easy", num_rooms: int = 10, seed: int = 42):
        mode_map = {
            "task_easy": "easy",
            "task_medium": "medium",
            "task_hard": "hard",
            "easy": "easy",
            "medium": "medium",
            "hard": "hard",
        }
        self.mode = mode_map.get(task_id, "easy")
        self.task_id = task_id
        self.env = HostelGridEnv(mode=self.mode, seed=seed)
        self.last_obs = None
        self.total_reward = 0.0
        self.steps = 0

    def reset(self) -> OpenEnvObservation:
        self.total_reward = 0.0
        self.steps = 0
        obs = self.env.reset()
        self.last_obs = obs
        return self._convert_obs(obs)

    def step(self, action: Action):
        room_actions = self._expand_action(action)
        obs, reward, done, info = self.env.step(RoomAction.from_list(room_actions))
        self.last_obs = obs
        self.total_reward += reward.total
        self.steps += 1

        return (
            self._convert_obs(obs),
            OpenEnvReward(value=float(reward.total), normalized=float(reward.normalized)),
            done,
            info,
        )

    def state(self) -> Dict[str, Any]:
        if self.last_obs is None:
            self.reset()
        return self._convert_obs(self.last_obs).model_dump()

    def score(self) -> float:
        if self.steps == 0:
            return 0.0
        avg = self.total_reward / self.steps
        return round(max(0.0, min(1.0, (avg + 2.0) / 3.0)), 4)

    def _expand_action(self, action: Action) -> List[int]:
        if action.room_actions is not None:
            if len(action.room_actions) != 10:
                raise ValueError("room_actions must contain exactly 10 actions")
            return action.room_actions

        global_action = 5 if action.action_id is None else action.action_id
        obs = self.env.state()

        if global_action == 0:
            return [7 if r.occupancy > 0.5 else 0 for r in obs.rooms]
        if global_action == 1:
            return [6 if r.occupancy > 0.5 else 0 for r in obs.rooms]
        if global_action == 2:
            return [
                0 if r.occupancy < 0.5 else int(r.ac * 4 + r.fan * 2 + r.light)
                for r in obs.rooms
            ]
        if global_action == 3:
            return [3 if r.occupancy > 0.5 else 0 for r in obs.rooms]
        if global_action == 4:
            return self._priority_safe_actions(obs)
        return [int(r.ac * 4 + r.fan * 2 + r.light) for r in obs.rooms]

    def _priority_safe_actions(self, obs) -> List[int]:
        actions = [0] * 10
        used = 0.0
        budget = obs.power_budget
        rooms = sorted(obs.rooms, key=lambda r: (-r.priority, -r.occupancy, -r.complaint_level))

        for r in rooms:
            if r.occupancy < 0.5:
                continue

            idx = r.room_id
            full_cost = 990
            partial_cost = 90

            if used + full_cost <= budget:
                actions[idx] = 7
                used += full_cost
            elif used + partial_cost <= budget:
                actions[idx] = 6
                used += partial_cost

        return actions

    def _convert_obs(self, obs) -> OpenEnvObservation:
        avg_occupancy = obs.occupied_rooms / max(1, len(obs.rooms))
        peak_hours = {9, 10, 11, 12, 13, 14, 18, 19, 20, 21}
        carbon_rate = 0.85 if obs.hour in peak_hours else 0.45
        tariff = 8.5 if obs.hour in peak_hours else 4.0

        return OpenEnvObservation(
            power_usage=float(obs.power_used / 1000.0),
            avg_occupancy=float(avg_occupancy),
            complaint_level=int(obs.total_complaints),
            time_of_day=int(obs.hour),
            carbon_rate=float(carbon_rate),
            current_cost=float((obs.power_used / 1000.0) * tariff),
            rooms=[r.to_dict() for r in obs.rooms],
            mode=obs.mode,
            heatwave=obs.heatwave,
        )


In [ ]:
%cd /content/hostalgrid-plus-plus

from env.openenv_api import HostelGridOpenEnv, Action

env = HostelGridOpenEnv(task_id="task_medium")
obs = env.reset()
print(obs.model_dump())

obs, reward, done, info = env.step(Action(action_id=4))
print("reward:", reward.value, "score:", env.score(), "done:", done)
print(info)


In [ ]:
import json, os, random

from env.hostelgrid_env import HostelGridEnv
from env.action import Action

os.makedirs("data", exist_ok=True)

def priority_policy(env):
    actions = [0] * 10
    used = 0
    budget = env.hostel.power_budget
    order = sorted(range(10), key=lambda i: (-env.hostel.rooms[i].priority, -env.hostel.rooms[i].occupancy))

    for i in order:
        room = env.hostel.rooms[i]
        if room.occupancy == 0:
            continue
        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90

    return actions

def make_prompt(obs):
    return (
        "You are EnergyMind, an LLM managing hostel electricity.\n"
        "Choose 10 room actions. Each action is 0-7:\n"
        "0 all_off, 1 ac_only, 2 fan_only, 3 light_only, 4 ac_fan, 5 ac_light, 6 fan_light, 7 all_on.\n"
        "Prioritize occupied high-priority rooms, reduce complaints, avoid wasting power.\n"
        f"Observation JSON:\n{json.dumps(obs.to_dict())}\n"
        "Return only JSON like: {\"actions\": [0,1,...], \"reason\": \"short reason\"}"
    )

rows = []
for mode in ["easy", "medium", "hard"]:
    for seed in range(50):
        env = HostelGridEnv(mode=mode, seed=seed)
        obs = env.reset()

        for step in range(30):
            actions = priority_policy(env)
            prompt = make_prompt(obs)
            completion = json.dumps({
                "actions": actions,
                "reason": "Serve occupied high-priority rooms first while keeping empty rooms off and staying within budget."
            })

            next_obs, reward, done, info = env.step(Action.from_list(actions))
            rows.append({
                "prompt": prompt,
                "completion": completion,
                "reward": reward.total,
                "mode": mode,
                "seed": seed,
                "step": step
            })
            obs = next_obs
            if done:
                break

with open("data/llm_trajectories.jsonl", "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

print("rows:", len(rows))
print("saved: data/llm_trajectories.jsonl")


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

dataset = load_dataset("json", data_files="data/llm_trajectories.jsonl", split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)

config = SFTConfig(
    output_dir="outputs/energymind-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    max_steps=80,
    logging_steps=10,
    save_steps=80,
    max_seq_length=2048,
    fp16=True,
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

dataset = load_dataset("json", data_files="data/llm_trajectories.jsonl", split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])

config = SFTConfig(
    output_dir="outputs/energymind-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    max_steps=80,
    logging_steps=10,
    save_steps=80,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q -U trl peft accelerate transformers datasets bitsandbytes


In [ ]:
import trl
import peft
import transformers
import datasets

print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

dataset = load_dataset("json", data_files="data/llm_trajectories.jsonl", split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])

config = SFTConfig(
    output_dir="outputs/energymind-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    max_steps=80,
    logging_steps=10,
    save_steps=80,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
%cd /content/hostalgrid-plus-plus
!pwd
!ls
!ls data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!mkdir -p /content/drive/MyDrive/hostelgrid-work
%cd /content/drive/MyDrive/hostelgrid-work


In [ ]:
!git clone https://github.com/anshu-ai-arch/hostalgrid-plus-plus.git
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus


In [ ]:
!pip install -r requirements.txt
!pip install -q -U trl peft accelerate transformers datasets bitsandbytes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import os

DATA_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories.jsonl"

print("Current folder:")
!pwd

print("\nDataset exists:", os.path.exists(DATA_PATH))
print("Dataset path:", DATA_PATH)

if os.path.exists(DATA_PATH):
    print("Dataset size MB:", round(os.path.getsize(DATA_PATH) / 1024 / 1024, 2))
else:
    print("Dataset missing. Rerun Cell 6 first.")


In [ ]:
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import json, os
from env.hostelgrid_env import HostelGridEnv
from env.action import Action

DATA_DIR = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data"
DATA_PATH = f"{DATA_DIR}/llm_trajectories.jsonl"

os.makedirs(DATA_DIR, exist_ok=True)

def priority_policy(env):
    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    order = sorted(
        range(10),
        key=lambda i: (
            -env.hostel.rooms[i].priority,
            -env.hostel.rooms[i].occupancy,
            -env.hostel.rooms[i].complaint,
        )
    )

    for i in order:
        room = env.hostel.rooms[i]

        if room.occupancy == 0:
            continue

        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90

    return actions

def make_prompt(obs):
    return (
        "You are EnergyMind, an LLM managing hostel electricity.\n"
        "Choose 10 room actions. Each action is 0-7:\n"
        "0 all_off, 1 ac_only, 2 fan_only, 3 light_only, "
        "4 ac_fan, 5 ac_light, 6 fan_light, 7 all_on.\n"
        "Prioritize occupied high-priority rooms, reduce complaints, "
        "avoid wasting power, and stay within budget.\n"
        f"Observation JSON:\n{json.dumps(obs.to_dict())}\n"
        "Return only JSON like: "
        "{\"actions\": [0,1,...], \"reason\": \"short reason\"}"
    )

rows_written = 0

with open(DATA_PATH, "w") as f:
    for mode in ["easy", "medium", "hard"]:
        for seed in range(50):
            env = HostelGridEnv(mode=mode, seed=seed)
            obs = env.reset()

            for step in range(30):
                actions = priority_policy(env)
                prompt = make_prompt(obs)

                completion = json.dumps({
                    "actions": actions,
                    "reason": (
                        "Serve occupied high-priority rooms first while keeping "
                        "empty rooms off and staying within budget."
                    )
                })

                next_obs, reward, done, info = env.step(Action.from_list(actions))

                row = {
                    "prompt": prompt,
                    "completion": completion,
                    "reward": float(reward.total),
                    "mode": mode,
                    "seed": seed,
                    "step": step,
                }

                f.write(json.dumps(row) + "\n")
                rows_written += 1

                obs = next_obs
                if done:
                    break

print("Rows written:", rows_written)
print("Saved dataset:", DATA_PATH)
print("Size MB:", round(os.path.getsize(DATA_PATH) / 1024 / 1024, 2))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import os

DATA_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories.jsonl"

print("Current folder:")
!pwd

print("\nDataset exists:", os.path.exists(DATA_PATH))
print("Dataset path:", DATA_PATH)

if os.path.exists(DATA_PATH):
    print("Dataset size MB:", round(os.path.getsize(DATA_PATH) / 1024 / 1024, 2))
else:
    print("Dataset missing. Rerun Cell 6 first.")


In [ ]:
!pip install -q -U trl peft accelerate transformers datasets bitsandbytes

import trl, peft, transformers, datasets
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)


In [ ]:
!pip install -q -U "torchao>=0.16.0" peft trl accelerate transformers

from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

DATA_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories.jsonl"

dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42).select(range(min(300, len(dataset))))

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-sft-quick",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=20,
    logging_steps=2,
    save_steps=20,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-sft-quick-lora"

trainer.model.save_pretrained(MODEL_DIR)
trainer.processing_class.save_pretrained(MODEL_DIR)

print("Saved model adapter to:", MODEL_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import os

MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-sft-quick-lora"
DATA_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories.jsonl"

print("Model exists:", os.path.exists(MODEL_DIR))
print("Dataset exists:", os.path.exists(DATA_PATH))

if os.path.exists(MODEL_DIR):
    print("Model files:", os.listdir(MODEL_DIR))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

!pip install -q -U transformers peft accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-sft-quick-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_model = PeftModel.from_pretrained(ft_base, LORA_DIR)
ft_model.eval()
base_model.eval()

print("Loaded base and fine-tuned LoRA model")


In [ ]:
import json
import re
import numpy as np

from env.hostelgrid_env import HostelGridEnv
from env.action import Action


def make_llm_prompt(obs):
    return (
        "You are EnergyMind, an LLM managing hostel electricity.\n"
        "Choose exactly 10 room actions. Each action is 0-7:\n"
        "0 all_off, 1 ac_only, 2 fan_only, 3 light_only, "
        "4 ac_fan, 5 ac_light, 6 fan_light, 7 all_on.\n"
        "Rules: prioritize occupied high-priority rooms, reduce complaints, "
        "avoid wasting power in empty rooms, stay within budget.\n"
        f"Observation JSON:\n{json.dumps(obs.to_dict())}\n"
        "Return only valid JSON with this schema:\n"
        "{\"actions\": [0,0,0,0,0,0,0,0,0,0], \"reason\": \"short reason\"}"
    )


def extract_actions(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return None, False

        data = json.loads(match.group(0))
        actions = data.get("actions")

        if not isinstance(actions, list):
            return None, False

        if len(actions) != 10:
            return None, False

        if not all(isinstance(a, int) and 0 <= a <= 7 for a in actions):
            return None, False

        return actions, True
    except Exception:
        return None, False


def generate_actions(model, obs, max_new_tokens=120):
    prompt = make_llm_prompt(obs)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    actions, valid = extract_actions(decoded)

    if not valid:
        actions = [0] * 10

    return actions, valid, decoded


def priority_heuristic_actions(env):
    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    order = sorted(
        range(10),
        key=lambda i: (
            -env.hostel.rooms[i].priority,
            -env.hostel.rooms[i].occupancy,
            -env.hostel.rooms[i].complaint,
        )
    )

    for i in order:
        room = env.hostel.rooms[i]
        if room.occupancy == 0:
            continue

        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90

    return actions


In [ ]:
def evaluate_policy(policy_name, policy_fn, mode="medium", seeds=[100, 200], episodes_per_seed=2, steps=20):
    rewards = []
    complaints = []
    hp_sats = []
    valid_rates = []

    for seed in seeds:
        env = HostelGridEnv(mode=mode, seed=seed)

        for _ in range(episodes_per_seed):
            obs = env.reset()
            total_reward = 0.0
            valid_count = 0
            step_count = 0
            last_info = None

            for _ in range(steps):
                actions, valid = policy_fn(env, obs)
                valid_count += int(valid)
                step_count += 1

                obs, reward, done, info = env.step(Action.from_list(actions))
                total_reward += reward.total
                last_info = info

                if done:
                    break

            rewards.append(total_reward)
            complaints.append(last_info["complaints"])
            hp_sats.append(last_info["hp_satisfied"])
            valid_rates.append(valid_count / max(1, step_count))

    return {
        "policy": policy_name,
        "mode": mode,
        "reward_mean": float(np.mean(rewards)),
        "reward_std": float(np.std(rewards)),
        "complaints_mean": float(np.mean(complaints)),
        "hp_sat_mean": float(np.mean(hp_sats)),
        "valid_json_rate": float(np.mean(valid_rates)),
    }


def base_policy(env, obs):
    actions, valid, text = generate_actions(base_model, obs)
    return actions, valid


def finetuned_policy(env, obs):
    actions, valid, text = generate_actions(ft_model, obs)
    return actions, valid


def heuristic_policy(env, obs):
    return priority_heuristic_actions(env), True


In [ ]:
results = []

results.append(evaluate_policy("Base LLM", base_policy, mode="medium"))
results.append(evaluate_policy("Fine-tuned LLM", finetuned_policy, mode="medium"))
results.append(evaluate_policy("Priority Heuristic", heuristic_policy, mode="medium"))

results


In [ ]:
results = []

results.append(
    evaluate_policy(
        "Base LLM",
        base_policy,
        mode="medium",
        seeds=[100],
        episodes_per_seed=1,
        steps=5,
    )
)

results.append(
    evaluate_policy(
        "Fine-tuned LLM",
        finetuned_policy,
        mode="medium",
        seeds=[100],
        episodes_per_seed=1,
        steps=5,
    )
)

results.append(
    evaluate_policy(
        "Priority Heuristic",
        heuristic_policy,
        mode="medium",
        seeds=[100],
        episodes_per_seed=1,
        steps=5,
    )
)

results


In [ ]:
env = HostelGridEnv(mode="medium", seed=100)
obs = env.reset()

base_actions, base_valid, base_text = generate_actions(base_model, obs)
ft_actions, ft_valid, ft_text = generate_actions(ft_model, obs)

print("BASE VALID:", base_valid)
print("BASE ACTIONS:", base_actions)
print("BASE TEXT:\n", base_text)

print("\n" + "="*80 + "\n")

print("FT VALID:", ft_valid)
print("FT ACTIONS:", ft_actions)
print("FT TEXT:\n", ft_text)


In [ ]:
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import json, os
from env.hostelgrid_env import HostelGridEnv
from env.action import Action

DATA_DIR = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data"
COMPACT_PATH = f"{DATA_DIR}/llm_trajectories_compact.jsonl"

os.makedirs(DATA_DIR, exist_ok=True)

def priority_policy(env):
    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    order = sorted(
        range(10),
        key=lambda i: (
            -env.hostel.rooms[i].priority,
            -env.hostel.rooms[i].occupancy,
            -env.hostel.rooms[i].complaint,
        )
    )

    for i in order:
        room = env.hostel.rooms[i]

        if room.occupancy == 0:
            actions[i] = 0
            continue

        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90
        else:
            actions[i] = 0

    return actions

def compact_prompt(env, obs):
    lines = []
    lines.append("Task: choose 10 room actions for hostel electricity control.")
    lines.append("Actions: 0=off, 6=fan_light, 7=all_on.")
    lines.append("Policy: occupied high-priority rooms should be served first; empty rooms must be off; stay within power budget.")
    lines.append(f"Mode={obs.mode}, hour={obs.hour}, heatwave={obs.heatwave}, power_budget={obs.power_budget}, power_used={obs.power_used}")

    for r in env.hostel.rooms:
        lines.append(
            f"room {r.room_id}: occ={r.occupancy}, priority={r.priority}, "
            f"complaint={r.complaint}, ac={r.ac}, fan={r.fan}, light={r.light}"
        )

    lines.append('Return only JSON: {"actions":[10 integers], "reason":"short"}')
    return "\n".join(lines)

rows = 0

with open(COMPACT_PATH, "w") as f:
    for mode in ["easy", "medium", "hard"]:
        for seed in range(120):
            env = HostelGridEnv(mode=mode, seed=seed)
            obs = env.reset()

            for step in range(40):
                actions = priority_policy(env)
                prompt = compact_prompt(env, obs)

                completion = json.dumps({
                    "actions": actions,
                    "reason": "serve occupied high-priority rooms first, keep empty rooms off, stay within budget"
                })

                next_obs, reward, done, info = env.step(Action.from_list(actions))

                f.write(json.dumps({
                    "prompt": prompt,
                    "completion": completion,
                    "reward": float(reward.total),
                    "mode": mode,
                    "seed": seed,
                    "step": step,
                    "actions": actions,
                }) + "\n")

                rows += 1
                obs = next_obs

                if done:
                    break

print("Rows:", rows)
print("Saved:", COMPACT_PATH)
print("Size MB:", round(os.path.getsize(COMPACT_PATH) / 1024 / 1024, 2))


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

COMPACT_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories_compact.jsonl"

dataset = load_dataset("json", data_files=COMPACT_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42).select(range(min(2000, len(dataset))))

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    max_steps=150,
    logging_steps=10,
    save_steps=150,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft-lora"

trainer.model.save_pretrained(MODEL_DIR)
trainer.processing_class.save_pretrained(MODEL_DIR)

print("Saved:", MODEL_DIR)


In [ ]:
import os

AUTO_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft"
print(os.path.exists(AUTO_DIR))

if os.path.exists(AUTO_DIR):
    for root, dirs, files in os.walk(AUTO_DIR):
        level = root.replace(AUTO_DIR, "").count(os.sep)
        if level < 2:
            print(root, files[:5])


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

COMPACT_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories_compact.jsonl"

dataset = load_dataset("json", data_files=COMPACT_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42).select(range(min(2000, len(dataset))))

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    max_steps=150,
    logging_steps=10,
    save_steps=150,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

!pip install -q -U trl peft accelerate transformers datasets bitsandbytes
!pip uninstall -y torchao


In [ ]:
import trl, peft, transformers, datasets

print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

!pip uninstall -y numpy torchao
!pip install -q --no-cache-dir "numpy==1.26.4"
!pip install -q --no-cache-dir "datasets==3.6.0" "transformers==4.56.2" "accelerate==1.10.1" "peft==0.17.1" "trl==0.23.0" "bitsandbytes"

print("Setup done. Now restart runtime once more if imports fail.")


In [ ]:
import numpy as np
import trl, peft, transformers, datasets

print("numpy:", np.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)


In [ ]:
!pip uninstall -y transformers trl peft accelerate datasets tokenizers huggingface_hub safetensors
!pip install -q --no-cache-dir --force-reinstall \
  "transformers==4.56.2" \
  "tokenizers==0.22.1" \
  "huggingface_hub==0.35.3" \
  "safetensors==0.6.2" \
  "datasets==3.6.0" \
  "accelerate==1.10.1" \
  "peft==0.17.1" \
  "trl==0.23.0"
!pip uninstall -y torchao


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import numpy as np
import transformers, trl, peft, datasets, accelerate

print("numpy:", np.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)


In [ ]:
!pip uninstall -y torchvision torchaudio torchao


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import torch
import transformers, trl, peft, datasets, accelerate

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import torch
import transformers, trl, peft, datasets, accelerate

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)


In [ ]:
import os

COMPACT_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories_compact.jsonl"

print("Compact dataset exists:", os.path.exists(COMPACT_PATH))

if os.path.exists(COMPACT_PATH):
    print("Size MB:", round(os.path.getsize(COMPACT_PATH) / 1024 / 1024, 2))
else:
    print("Missing compact dataset. Regenerate it before training.")


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

COMPACT_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories_compact.jsonl"

dataset = load_dataset("json", data_files=COMPACT_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42).select(range(min(1000, len(dataset))))

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    max_steps=80,
    logging_steps=10,
    save_steps=25,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

COMPACT_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_trajectories_compact.jsonl"

dataset = load_dataset("json", data_files=COMPACT_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42).select(range(min(1000, len(dataset))))

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    max_steps=80,
    logging_steps=10,
    save_steps=25,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft-lora"

trainer.model.save_pretrained(MODEL_DIR)
trainer.processing_class.save_pretrained(MODEL_DIR)

print("Saved:", MODEL_DIR)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-compact-sft-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_model = PeftModel.from_pretrained(ft_base, LORA_DIR)
ft_model.eval()
base_model.eval()

print("Loaded base model and compact fine-tuned model.")


In [ ]:
import json
import re
import numpy as np

from env.hostelgrid_env import HostelGridEnv
from env.action import Action


def make_compact_prompt(env, obs):
    lines = []
    lines.append("Task: choose 10 room actions for hostel electricity control.")
    lines.append("Actions: 0=off, 6=fan_light, 7=all_on.")
    lines.append("Policy: occupied high-priority rooms should be served first; empty rooms must be off; stay within power budget.")
    lines.append(f"Mode={obs.mode}, hour={obs.hour}, heatwave={obs.heatwave}, power_budget={obs.power_budget}, power_used={obs.power_used}")

    for r in env.hostel.rooms:
        lines.append(
            f"room {r.room_id}: occ={r.occupancy}, priority={r.priority}, "
            f"complaint={r.complaint}, ac={r.ac}, fan={r.fan}, light={r.light}"
        )

    lines.append('Return only JSON: {"actions":[10 integers], "reason":"short"}')
    return "\n".join(lines)


def extract_actions(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return None, False

        data = json.loads(match.group(0))
        actions = data.get("actions")

        if not isinstance(actions, list):
            return None, False

        if len(actions) != 10:
            return None, False

        if not all(isinstance(a, int) and 0 <= a <= 7 for a in actions):
            return None, False

        return actions, True
    except Exception:
        return None, False


def generate_actions(model, env, obs, max_new_tokens=80):
    prompt = make_compact_prompt(env, obs)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    actions, valid = extract_actions(decoded)

    if not valid:
        actions = [0] * 10

    return actions, valid, decoded


def priority_heuristic_actions(env):
    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    order = sorted(
        range(10),
        key=lambda i: (
            -env.hostel.rooms[i].priority,
            -env.hostel.rooms[i].occupancy,
            -env.hostel.rooms[i].complaint,
        )
    )

    for i in order:
        room = env.hostel.rooms[i]
        if room.occupancy == 0:
            continue

        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90

    return actions


In [ ]:
env = HostelGridEnv(mode="medium", seed=100)
obs = env.reset()

base_actions, base_valid, base_text = generate_actions(base_model, env, obs)
ft_actions, ft_valid, ft_text = generate_actions(ft_model, env, obs)

print("BASE VALID:", base_valid)
print("BASE ACTIONS:", base_actions)
print("BASE TEXT:\n", base_text)

print("\n" + "="*80 + "\n")

print("FT VALID:", ft_valid)
print("FT ACTIONS:", ft_actions)
print("FT TEXT:\n", ft_text)

print("\n" + "="*80 + "\n")

print("HEURISTIC ACTIONS:", priority_heuristic_actions(env))


In [ ]:
def evaluate_policy(policy_name, policy_fn, mode="medium", seeds=[100], episodes_per_seed=1, steps=5):
    rewards = []
    complaints = []
    hp_sats = []
    valid_rates = []

    for seed in seeds:
        env = HostelGridEnv(mode=mode, seed=seed)

        for _ in range(episodes_per_seed):
            obs = env.reset()
            total_reward = 0.0
            valid_count = 0
            step_count = 0
            last_info = None

            for _ in range(steps):
                actions, valid = policy_fn(env, obs)
                valid_count += int(valid)
                step_count += 1

                obs, reward, done, info = env.step(Action.from_list(actions))
                total_reward += reward.total
                last_info = info

                if done:
                    break

            rewards.append(total_reward)
            complaints.append(last_info["complaints"])
            hp_sats.append(last_info["hp_satisfied"])
            valid_rates.append(valid_count / max(1, step_count))

    return {
        "policy": policy_name,
        "mode": mode,
        "reward_mean": float(np.mean(rewards)),
        "reward_std": float(np.std(rewards)),
        "complaints_mean": float(np.mean(complaints)),
        "hp_sat_mean": float(np.mean(hp_sats)),
        "valid_json_rate": float(np.mean(valid_rates)),
    }


def base_policy(env, obs):
    actions, valid, text = generate_actions(base_model, env, obs)
    return actions, valid


def finetuned_policy(env, obs):
    actions, valid, text = generate_actions(ft_model, env, obs)
    return actions, valid


def heuristic_policy(env, obs):
    return priority_heuristic_actions(env), True


results = []
results.append(evaluate_policy("Base LLM", base_policy))
results.append(evaluate_policy("Fine-tuned LLM", finetuned_policy))
results.append(evaluate_policy("Priority Heuristic", heuristic_policy))

results


In [ ]:
%cd /content
!ls
!pwd
%cd /content/hostalgrid-plus-plus-anshu
!pwd
!ls
!git branch
!git status

In [ ]:
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import json, os
from env.hostelgrid_env import HostelGridEnv

DATA_DIR = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data"
ROOM_RULE_PATH = f"{DATA_DIR}/llm_room_rule.jsonl"

os.makedirs(DATA_DIR, exist_ok=True)

def room_action(room, used, budget):
    if room.occupancy == 0:
        return 0, used

    if used + 990 <= budget:
        return 7, used + 990

    if used + 90 <= budget:
        return 6, used + 90

    return 0, used

def expert_actions(env):
    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    order = sorted(
        range(10),
        key=lambda i: (
            -env.hostel.rooms[i].priority,
            -env.hostel.rooms[i].occupancy,
            -env.hostel.rooms[i].complaint,
        )
    )

    for i in order:
        action, used = room_action(env.hostel.rooms[i], used, budget)
        actions[i] = action

    return actions

def prompt_for_state(env):
    lines = []
    lines.append("Return actions for 10 rooms as JSON only.")
    lines.append("Use action 7 for occupied rooms if budget allows.")
    lines.append("Use action 6 if only fan_light budget allows.")
    lines.append("Use action 0 for empty rooms.")
    lines.append("Serve higher priority first.")
    lines.append(f"budget={env.hostel.power_budget}")

    for r in env.hostel.rooms:
        lines.append(f"room {r.room_id}: occupied={r.occupancy}, priority={r.priority}, complaint={r.complaint}")

    lines.append('JSON only: {"actions":[a0,a1,a2,a3,a4,a5,a6,a7,a8,a9]}')
    return "\n".join(lines)

rows = 0

with open(ROOM_RULE_PATH, "w") as f:
    for mode in ["easy", "medium", "hard"]:
        for seed in range(500):
            env = HostelGridEnv(mode=mode, seed=seed)
            env.reset()

            actions = expert_actions(env)
            prompt = prompt_for_state(env)
            completion = json.dumps({"actions": actions})

            f.write(json.dumps({
                "prompt": prompt,
                "completion": completion,
                "mode": mode,
                "actions": actions,
            }) + "\n")
            rows += 1

print("Rows:", rows)
print("Saved:", ROOM_RULE_PATH)


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

ROOM_RULE_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_room_rule.jsonl"

dataset = load_dataset("json", data_files=ROOM_RULE_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42)

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-room-rule-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    max_steps=250,
    logging_steps=25,
    save_steps=50,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-room-rule-lora"

trainer.model.save_pretrained(MODEL_DIR)
trainer.processing_class.save_pretrained(MODEL_DIR)

print("Saved:", MODEL_DIR)


In [ ]:
def make_room_rule_prompt(env, obs):
    lines = []
    lines.append("Return actions for 10 rooms as JSON only.")
    lines.append("Use action 7 for occupied rooms if budget allows.")
    lines.append("Use action 6 if only fan_light budget allows.")
    lines.append("Use action 0 for empty rooms.")
    lines.append("Serve higher priority first.")
    lines.append(f"budget={env.hostel.power_budget}")

    for r in env.hostel.rooms:
        lines.append(f"room {r.room_id}: occupied={r.occupancy}, priority={r.priority}, complaint={r.complaint}")

    lines.append('JSON only: {"actions":[a0,a1,a2,a3,a4,a5,a6,a7,a8,a9]}')
    return "\n".join(lines)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-room-rule-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_model = PeftModel.from_pretrained(ft_base, LORA_DIR)
base_model.eval()
ft_model.eval()

print("Loaded room-rule fine-tuned model.")


In [ ]:
import json
import re
import numpy as np

from env.hostelgrid_env import HostelGridEnv
from env.action import Action


def make_room_rule_prompt(env, obs):
    lines = []
    lines.append("Return actions for 10 rooms as JSON only.")
    lines.append("Use action 7 for occupied rooms if budget allows.")
    lines.append("Use action 6 if only fan_light budget allows.")
    lines.append("Use action 0 for empty rooms.")
    lines.append("Serve higher priority first.")
    lines.append(f"budget={env.hostel.power_budget}")

    for r in env.hostel.rooms:
        lines.append(
            f"room {r.room_id}: occupied={r.occupancy}, "
            f"priority={r.priority}, complaint={r.complaint}"
        )

    lines.append('JSON only: {"actions":[a0,a1,a2,a3,a4,a5,a6,a7,a8,a9]}')
    return "\n".join(lines)


def extract_actions(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return None, False

        data = json.loads(match.group(0))
        actions = data.get("actions")

        if not isinstance(actions, list):
            return None, False

        if len(actions) != 10:
            return None, False

        if not all(isinstance(a, int) and 0 <= a <= 7 for a in actions):
            return None, False

        return actions, True
    except Exception:
        return None, False


def generate_actions(model, env, obs, max_new_tokens=60):
    prompt = make_room_rule_prompt(env, obs)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    actions, valid = extract_actions(decoded)

    if not valid:
        actions = [0] * 10

    return actions, valid, decoded


def priority_heuristic_actions(env):
    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    order = sorted(
        range(10),
        key=lambda i: (
            -env.hostel.rooms[i].priority,
            -env.hostel.rooms[i].occupancy,
            -env.hostel.rooms[i].complaint,
        )
    )

    for i in order:
        room = env.hostel.rooms[i]
        if room.occupancy == 0:
            continue

        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90

    return actions


In [ ]:
env = HostelGridEnv(mode="medium", seed=100)
obs = env.reset()

base_actions, base_valid, base_text = generate_actions(base_model, env, obs)
ft_actions, ft_valid, ft_text = generate_actions(ft_model, env, obs)
heuristic_actions = priority_heuristic_actions(env)

print("BASE VALID:", base_valid)
print("BASE ACTIONS:", base_actions)
print("BASE TEXT:\n", base_text)

print("\n" + "="*80 + "\n")

print("FT VALID:", ft_valid)
print("FT ACTIONS:", ft_actions)
print("FT TEXT:\n", ft_text)

print("\n" + "="*80 + "\n")

print("HEURISTIC ACTIONS:", heuristic_actions)
print("MATCH COUNT:", sum(a == b for a, b in zip(ft_actions, heuristic_actions)), "/ 10")


In [ ]:
def evaluate_policy(policy_name, policy_fn, mode="medium", seeds=[100], episodes_per_seed=1, steps=5):
    rewards = []
    complaints = []
    hp_sats = []
    valid_rates = []

    for seed in seeds:
        env = HostelGridEnv(mode=mode, seed=seed)

        for _ in range(episodes_per_seed):
            obs = env.reset()
            total_reward = 0.0
            valid_count = 0
            step_count = 0
            last_info = None

            for _ in range(steps):
                actions, valid = policy_fn(env, obs)
                valid_count += int(valid)
                step_count += 1

                obs, reward, done, info = env.step(Action.from_list(actions))
                total_reward += reward.total
                last_info = info

                if done:
                    break

            rewards.append(total_reward)
            complaints.append(last_info["complaints"])
            hp_sats.append(last_info["hp_satisfied"])
            valid_rates.append(valid_count / max(1, step_count))

    return {
        "policy": policy_name,
        "mode": mode,
        "reward_mean": float(np.mean(rewards)),
        "reward_std": float(np.std(rewards)),
        "complaints_mean": float(np.mean(complaints)),
        "hp_sat_mean": float(np.mean(hp_sats)),
        "valid_json_rate": float(np.mean(valid_rates)),
    }


def base_policy(env, obs):
    actions, valid, text = generate_actions(base_model, env, obs)
    return actions, valid


def finetuned_policy(env, obs):
    actions, valid, text = generate_actions(ft_model, env, obs)
    return actions, valid


def heuristic_policy(env, obs):
    return priority_heuristic_actions(env), True


results = []
results.append(evaluate_policy("Base LLM", base_policy))
results.append(evaluate_policy("Fine-tuned Room-Rule LLM", finetuned_policy))
results.append(evaluate_policy("Priority Heuristic", heuristic_policy))

results


In [ ]:
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus

import json, os, re
import numpy as np

from env.hostelgrid_env import HostelGridEnv
from env.action import Action


STRATEGIES = [
    "priority_safe",
    "complaint_rescue",
    "energy_saver",
    "comfort_all",
    "shutdown_empty",
    "do_nothing",
]


def current_actions(env):
    actions = []
    for r in env.hostel.rooms:
        action = int(r.ac * 4 + r.fan * 2 + r.light)
        actions.append(action)
    return actions


def execute_strategy(env, strategy):
    strategy = strategy if strategy in STRATEGIES else "priority_safe"

    if strategy == "do_nothing":
        return current_actions(env)

    if strategy == "comfort_all":
        return [7 if r.occupancy == 1 else 0 for r in env.hostel.rooms]

    if strategy == "energy_saver":
        return [6 if r.occupancy == 1 else 0 for r in env.hostel.rooms]

    if strategy == "shutdown_empty":
        actions = current_actions(env)
        for i, r in enumerate(env.hostel.rooms):
            if r.occupancy == 0:
                actions[i] = 0
        return actions

    actions = [0] * 10
    used = 0.0
    budget = env.hostel.power_budget

    if strategy == "complaint_rescue":
        order = sorted(
            range(10),
            key=lambda i: (
                -env.hostel.rooms[i].complaint,
                -env.hostel.rooms[i].priority,
                -env.hostel.rooms[i].occupancy,
            )
        )
    else:
        order = sorted(
            range(10),
            key=lambda i: (
                -env.hostel.rooms[i].priority,
                -env.hostel.rooms[i].occupancy,
                -env.hostel.rooms[i].complaint,
            )
        )

    for i in order:
        room = env.hostel.rooms[i]
        if room.occupancy == 0:
            continue

        if used + 990 <= budget:
            actions[i] = 7
            used += 990
        elif used + 90 <= budget:
            actions[i] = 6
            used += 90

    return actions


def expert_strategy(env):
    total_complaints = sum(r.complaint for r in env.hostel.rooms)
    occupied = sum(r.occupancy for r in env.hostel.rooms)
    power_ratio = env.hostel.power_ratio()

    if total_complaints >= 12:
        return "complaint_rescue"

    if power_ratio < 0.25:
        return "energy_saver"

    if occupied <= 3 and env.hostel.mode == "easy":
        return "comfort_all"

    return "priority_safe"


print("Strategy executor ready.")


In [ ]:
def make_strategy_prompt(env):
    lines = []
    lines.append("You are EnergyMind, an LLM planner for hostel electricity control.")
    lines.append("Choose exactly one strategy.")
    lines.append("Valid strategies: priority_safe, complaint_rescue, energy_saver, comfort_all, shutdown_empty, do_nothing.")
    lines.append("Return only JSON like: {\"strategy\":\"priority_safe\"}")
    lines.append(f"mode={env.hostel.mode}, hour={env.hostel.hour}, heatwave={env.hostel.heatwave}")
    lines.append(f"power_used={env.hostel.total_power()}, power_budget={env.hostel.power_budget}, power_ratio={env.hostel.power_ratio():.3f}")

    for r in env.hostel.rooms:
        lines.append(
            f"room {r.room_id}: occupied={r.occupancy}, priority={r.priority}, "
            f"complaint={r.complaint}, ac={r.ac}, fan={r.fan}, light={r.light}"
        )

    return "\n".join(lines)


def parse_strategy(text):
    try:
        text = text.replace("```json", "").replace("```", "").strip()
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return "priority_safe", False

        data = json.loads(match.group(0))
        strategy = data.get("strategy", "")

        if strategy not in STRATEGIES:
            return "priority_safe", False

        return strategy, True
    except Exception:
        return "priority_safe", False


print(make_strategy_prompt(HostelGridEnv(mode="medium", seed=100)))


In [ ]:
DATA_DIR = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data"
STRATEGY_PATH = f"{DATA_DIR}/llm_strategy_dataset.jsonl"

os.makedirs(DATA_DIR, exist_ok=True)

rows = 0

with open(STRATEGY_PATH, "w") as f:
    for mode in ["easy", "medium", "hard"]:
        for seed in range(300):
            env = HostelGridEnv(mode=mode, seed=seed)
            env.reset()

            for step in range(25):
                strategy = expert_strategy(env)
                prompt = make_strategy_prompt(env)
                completion = json.dumps({"strategy": strategy})

                actions = execute_strategy(env, strategy)
                obs, reward, done, info = env.step(Action.from_list(actions))

                f.write(json.dumps({
                    "prompt": prompt,
                    "completion": completion,
                    "strategy": strategy,
                    "mode": mode,
                    "reward": float(reward.total),
                    "step": step,
                }) + "\n")

                rows += 1
                if done:
                    break

print("Rows:", rows)
print("Saved:", STRATEGY_PATH)
print("Size MB:", round(os.path.getsize(STRATEGY_PATH) / 1024 / 1024, 2))


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

STRATEGY_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_strategy_dataset.jsonl"

dataset = load_dataset("json", data_files=STRATEGY_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42).select(range(min(3000, len(dataset))))

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-strategy-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=4e-4,
    max_steps=120,
    logging_steps=20,
    save_steps=40,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.02,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-strategy-lora"

trainer.model.save_pretrained(MODEL_DIR)
trainer.processing_class.save_pretrained(MODEL_DIR)

print("Saved:", MODEL_DIR)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-strategy-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

strategy_model = PeftModel.from_pretrained(ft_base, LORA_DIR)
base_model.eval()
strategy_model.eval()

print("Loaded strategy model.")


In [ ]:
def generate_strategy(model, env, max_new_tokens=40):
    prompt = make_strategy_prompt(env)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    strategy, valid = parse_strategy(decoded)
    return strategy, valid, decoded


env = HostelGridEnv(mode="medium", seed=100)
env.reset()

base_strategy, base_valid, base_text = generate_strategy(base_model, env)
ft_strategy, ft_valid, ft_text = generate_strategy(strategy_model, env)
expert = expert_strategy(env)

print("BASE:", base_valid, base_strategy, repr(base_text))
print("FT:", ft_valid, ft_strategy, repr(ft_text))
print("EXPERT:", expert)
print("FT ACTIONS:", execute_strategy(env, ft_strategy))
print("EXPERT ACTIONS:", execute_strategy(env, expert))


In [ ]:
def evaluate_strategy_policy(policy_name, strategy_fn, mode="medium", seeds=[100], episodes_per_seed=1, steps=10):
    rewards = []
    complaints = []
    hp_sats = []
    valid_rates = []

    for seed in seeds:
        env = HostelGridEnv(mode=mode, seed=seed)

        for _ in range(episodes_per_seed):
            env.reset()
            total_reward = 0.0
            valid_count = 0
            step_count = 0
            last_info = None

            for _ in range(steps):
                strategy, valid = strategy_fn(env)
                actions = execute_strategy(env, strategy)

                valid_count += int(valid)
                step_count += 1

                obs, reward, done, info = env.step(Action.from_list(actions))
                total_reward += reward.total
                last_info = info

                if done:
                    break

            rewards.append(total_reward)
            complaints.append(last_info["complaints"])
            hp_sats.append(last_info["hp_satisfied"])
            valid_rates.append(valid_count / max(1, step_count))

    return {
        "policy": policy_name,
        "reward_mean": float(np.mean(rewards)),
        "complaints_mean": float(np.mean(complaints)),
        "hp_sat_mean": float(np.mean(hp_sats)),
        "valid_strategy_rate": float(np.mean(valid_rates)),
    }


def base_strategy_policy(env):
    strategy, valid, text = generate_strategy(base_model, env)
    return strategy, valid


def ft_strategy_policy(env):
    strategy, valid, text = generate_strategy(strategy_model, env)
    return strategy, valid


def expert_strategy_policy(env):
    return expert_strategy(env), True


results = []
results.append(evaluate_strategy_policy("Base LLM Strategy", base_strategy_policy))
results.append(evaluate_strategy_policy("Fine-tuned LLM Strategy", ft_strategy_policy))
results.append(evaluate_strategy_policy("Expert Strategy", expert_strategy_policy))

results


In [ ]:
import json
from collections import Counter

STRATEGY_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_strategy_dataset.jsonl"

counts = Counter()

with open(STRATEGY_PATH) as f:
    for line in f:
        row = json.loads(line)
        counts[row["strategy"]] += 1

counts


In [ ]:
import json, random, os
from collections import defaultdict

STRATEGY_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_strategy_dataset.jsonl"
BALANCED_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_strategy_balanced.jsonl"

buckets = defaultdict(list)

with open(STRATEGY_PATH) as f:
    for line in f:
        row = json.loads(line)
        buckets[row["strategy"]].append(row)

for k, v in buckets.items():
    print(k, len(v))

min_count = min(len(v) for v in buckets.values())
target = min(1200, min_count)

rows = []
for strategy, items in buckets.items():
    rows.extend(random.sample(items, min(target, len(items))))

random.shuffle(rows)

with open(BALANCED_PATH, "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

print("Balanced rows:", len(rows))
print("Saved:", BALANCED_PATH)


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

BALANCED_PATH = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/data/llm_strategy_balanced.jsonl"

dataset = load_dataset("json", data_files=BALANCED_PATH, split="train")

def format_example(example):
    return {
        "text": (
            "<|user|>\n" + example["prompt"] +
            "\n<|assistant|>\n" + example["completion"]
        )
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
dataset = dataset.shuffle(seed=42)

config = SFTConfig(
    output_dir="/content/drive/MyDrive/hostelgrid-work/outputs/energymind-strategy-balanced-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    max_steps=160,
    logging_steps=20,
    save_steps=40,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=config,
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.02,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

trainer.train()


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-strategy-balanced-lora"

trainer.model.save_pretrained(MODEL_DIR)
trainer.processing_class.save_pretrained(MODEL_DIR)

print("Saved:", MODEL_DIR)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_DIR = "/content/drive/MyDrive/hostelgrid-work/outputs/energymind-strategy-balanced-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

strategy_model = PeftModel.from_pretrained(ft_base, LORA_DIR)
base_model.eval()
strategy_model.eval()

print("Loaded balanced strategy model.")


In [ ]:
test_cases = [
    ("easy", 100),
    ("medium", 100),
    ("medium", 200),
    ("hard", 100),
    ("hard", 999),
]

for mode, seed in test_cases:
    env = HostelGridEnv(mode=mode, seed=seed)
    env.reset()

    ft_strategy, ft_valid, ft_text = generate_strategy(strategy_model, env)
    expert = expert_strategy(env)

    print("="*60)
    print("mode:", mode, "seed:", seed)
    print("FT:", ft_valid, ft_strategy, repr(ft_text))
    print("EXPERT:", expert)
    print("MATCH:", ft_strategy == expert)


In [ ]:
def ft_balanced_strategy_policy(env):
    strategy, valid, text = generate_strategy(strategy_model, env)
    return strategy, valid


results = []
results.append(evaluate_strategy_policy("Base LLM Strategy", base_strategy_policy, seeds=[100, 200, 300], episodes_per_seed=1, steps=10))
results.append(evaluate_strategy_policy("Fine-tuned Balanced Strategy", ft_balanced_strategy_policy, seeds=[100, 200, 300], episodes_per_seed=1, steps=10))
results.append(evaluate_strategy_policy("Expert Strategy", expert_strategy_policy, seeds=[100, 200, 300], episodes_per_seed=1, steps=10))

results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

results = [
    {
        "policy": "Base LLM Strategy Planner",
        "reward_mean": 3.2424447111134334,
        "complaints_mean": 1.3333333333333333,
        "hp_sat_mean": 3.0,
        "valid_strategy_rate": 1.0,
    },
    {
        "policy": "Fine-tuned SFT Strategy",
        "reward_mean": -0.46359075662634236,
        "complaints_mean": 57.333333333333336,
        "hp_sat_mean": 2.0,
        "valid_strategy_rate": 1.0,
    },
    {
        "policy": "Rule Expert Strategy",
        "reward_mean": 2.8842653715339703,
        "complaints_mean": 6.333333333333333,
        "hp_sat_mean": 3.0,
        "valid_strategy_rate": 1.0,
    },
]

df = pd.DataFrame(results)
display(df)

OUT_DIR = "/content/drive/MyDrive/hostelgrid-work/eval_results"
PLOT_DIR = "/content/drive/MyDrive/hostelgrid-work/plots"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

csv_path = f"{OUT_DIR}/strategy_planner_results.csv"
df.to_csv(csv_path, index=False)

plt.figure(figsize=(9, 5))
plt.bar(df["policy"], df["reward_mean"])
plt.title("HostelGrid++ Strategy Planner Reward")
plt.ylabel("Mean Reward")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/strategy_planner_reward.png", dpi=150)
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(df["policy"], df["complaints_mean"])
plt.title("HostelGrid++ Strategy Planner Complaints")
plt.ylabel("Mean Complaints")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/strategy_planner_complaints.png", dpi=150)
plt.show()

print("Saved:", csv_path)
print("Plots:", PLOT_DIR)


In [ ]:
!find /content -maxdepth 3 -type d -name ".git"

In [ ]:
%cd /content/hostalgrid-plus-plus
!git status
!git branch --show-current

In [ ]:
%cd /content/drive/MyDrive/hostalgrid-work/hostalgrid-plus-plus
!git status


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if '.git' in dirs:
        print(root)


In [ ]:
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus
!git status


In [ ]:
from google.colab import drive
drive.flush_and_unmount()


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



In [ ]:
%cd /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus
!pwd
!git status
!git branch
!git remote -v


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
!ls "/content/drive/MyDrive/hostalgrid-work/hostalgrid-plus-plus"


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!find /content/drive/MyDrive -maxdepth 4 -type d -name "hostalgrid-plus-plus"


In [ ]:
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" status
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" branch -a
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" remote -v


In [ ]:
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" checkout ujjwal-feature


In [ ]:
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" status
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" diff --stat


In [ ]:
!find "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" -maxdepth 3 -type f | sort


In [ ]:
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" diff origin/ujjwal-feature -- env/openenv_api.py


In [ ]:
!git -C "/content/drive/MyDrive/hostalgrid-work/hostalgrid-plus-plus" diff origin/ujjwal-feature -- app.py


In [ ]:
!git -C "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus" diff origin/ujjwal-feature -- server/app.py


In [ ]:
!git -C "/content/drive/MyDrive/hostalgrid-work/hostalgrid-plus-plus" add data
!git -C "/content/drive/MyDrive/hostalgrid-work/hostalgrid-plus-plus" commit -m "Add generated LLM training datasets"


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!find /content/drive/MyDrive -maxdepth 4 -type d -name "hostalgrid-plus-plus"


In [ ]:
REPO = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus"

!ls "$REPO"
!git -C "$REPO" status
!git -C "$REPO" branch -a
!git -C "$REPO" remote -v


In [ ]:
!git -C "$REPO" fetch origin
!git -C "$REPO" checkout ujjwal-feature
!git -C "$REPO" status


In [ ]:
!git -C "$REPO" status
!git -C "$REPO" diff --stat


In [ ]:
REPO = "/content/drive/MyDrive/hostalgrid-work/hostalgrid-plus-plus"

!git -C "$REPO" checkout ujjwal-feature
!git -C "$REPO" add data
!git -C "$REPO" status


In [ ]:
!sed -n '1,220p' "$REPO/env/openenv_api.py"
!sed -n '1,80p' "$REPO/app.py"


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

repo_paths = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if '.git' in dirs and root.endswith('hostalgrid-plus-plus'):
        repo_paths.append(root)

print(repo_paths)


In [ ]:
REPO = repo_paths[0]
print("Using repo:", REPO)

!ls "$REPO"
!git -C "$REPO" status


In [ ]:
!sed -n '1,220p' "$REPO/env/openenv_api.py"
!sed -n '1,80p' "$REPO/app.py"


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus"
!find /content/drive/MyDrive -maxdepth 4 -type d -name "hostalgrid-plus-plus"
!ls "$REPO"
!git -C "$REPO" status


In [ ]:
%%writefile /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/env/openenv_api.py
# paste the full final openenv_api.py here
# env/openenv_api.py
# Fixed:
# 1. Observation model now includes battery_level (was silently dropped)
# 2. _vec_to_obs updated for 15-feature vector (was 14)
# 3. demand_supply_ratio denormalization corrected (* 3.0, matching to_vector)
# 4. system_trust and battery_level passed through from info dict each step
# 5. score() thresholds recalibrated to realistic episode ranges

from pydantic import BaseModel
from typing import Any, Dict, Optional
import numpy as np
import random

from env.hostelgrid_env import HostelGridEnv
from env.state import EpisodeState


# ── Typed Models (OpenEnv spec) ───────────────────────────────

class Observation(BaseModel):
    power_usage:          float
    avg_temperature:      float
    avg_occupancy:        float
    complaint_level:      int
    time_of_day:          int
    carbon_rate:          float
    current_cost:         float
    system_trust:         float
    peak_hour:            bool
    solar_output:         float
    battery_level:        float    # FIX: was missing from model
    fairness_score:       float
    demand_supply_ratio:  float
    violations_this_step: int


class Action(BaseModel):
    action_id: int   # 0-9


class Reward(BaseModel):
    value:     float
    breakdown: Dict[str, float]
    done:      bool
    info:      Dict[str, Any]


# ── OpenEnv-compliant Environment ─────────────────────────────

class HostelGridOpenEnv:
    """
    OpenEnv spec-compliant wrapper.

    Fixes in this version:
    1. battery_level added to Observation model
    2. _vec_to_obs handles 15-feature vector (was 14)
    3. system_trust and battery_level read from info dict (env now writes them)
    4. demand_supply_ratio denormalized correctly (* 3.0)
    5. violations_this_step decoded from vec[14]
    6. EpisodeState.update() now receives system_trust for trust tracking
    """

    def __init__(self, task_id: str = "task_easy", num_rooms: int = 20):
        self.task_id       = task_id
        self.num_rooms     = num_rooms
        self._env          = HostelGridEnv(num_rooms=num_rooms, episode_hours=24)
        self._obs_vec      = None
        self._ep_state     = EpisodeState()

    def reset(self) -> Observation:
        self._obs_vec  = self._env.reset()
        self._ep_state = EpisodeState()
        return self._vec_to_obs(self._obs_vec)

    def step(self, action: Action):
        obs_vec, reward_val, done, info = self._env.step(action.action_id)
        self._obs_vec = obs_vec

        self._ep_state.update(
            reward         = reward_val,
            cost           = info.get("cost", 0),
            carbon         = info.get("carbon_rate", 0) * info.get("power", 0),
            complaints     = info.get("complaints", 0),
            violations     = info.get("violations", 0),
            demand_sat     = info.get("demand_sat", 0.0),
            fairness       = info.get("fairness", 1.0),
            peak_violation = info.get("peak_hour", False),
            hour           = info.get("hour", 0),
            # FIX: pass system_trust so EpisodeState can track it accurately
            system_trust   = info.get("system_trust", 1.0),
        )

        observation = self._vec_to_obs(obs_vec, info)
        reward_obj  = Reward(
            value    = reward_val,
            breakdown = {
                "power"         : info.get("power", 0),
                "complaints"    : info.get("complaints", 0),
                "cost"          : info.get("cost", 0),
                "violations"    : info.get("violations", 0),
                "demand_sat"    : info.get("demand_sat", 0),
                "fairness"      : info.get("fairness", 1.0),
                "system_trust"  : info.get("system_trust", 1.0),
                "battery_level" : info.get("battery_level", 0.5),
            },
            done = done,
            info = info,
        )

        return observation, reward_obj, done, info

    def state(self) -> Dict[str, Any]:
        ep = self._ep_state.summary()
        return {
            "task_id"             : self.task_id,
            "step"                : self._ep_state.steps,
            "done"                : self._ep_state.steps >= 24,
            "observation"         : self._vec_to_obs(
                                        self._obs_vec
                                    ).model_dump() if self._obs_vec is not None else {},
            "total_reward"        : ep["total_reward"],
            "total_cost"          : ep["total_cost"],
            "total_complaints"    : ep["total_complaints"],
            "total_violations"    : ep["total_violations"],
            "demand_satisfaction" : ep["demand_satisfaction"],
            "system_trust"        : ep["system_trust"],
            "avg_fairness"        : ep["avg_fairness"],
            "collapsed"           : ep["collapsed"],
        }

    def score(self) -> float:
        ep = self._ep_state.summary()

        if self.task_id == "task_easy":
            return self._score_easy(ep)
        elif self.task_id == "task_medium":
            return self._score_medium(ep)
        elif self.task_id == "task_hard":
            return self._score_hard(ep)
        return 0.0

    def _score_easy(self, ep: dict) -> float:
        score = 0.0

        tr = ep["total_reward"]
        if tr > 3.0:   score += 0.25
        elif tr > 0:   score += 0.10

        c = ep["total_cost"]
        if c < 1200:   score += 0.25
        elif c < 1500: score += 0.10

        cp = ep["total_complaints"]
        if cp < 40:    score += 0.25
        elif cp < 70:  score += 0.10

        v = ep["total_violations"]
        if v == 0:     score += 0.25
        elif v < 10:   score += 0.10

        return round(min(score, 1.0), 4)

    def _score_medium(self, ep: dict) -> float:
        score = 0.0

        tr = ep["total_reward"]
        if tr > 4.0:   score += 0.20
        elif tr > 0:   score += 0.10

        c = ep["total_cost"]
        if c < 1000:   score += 0.20
        elif c < 1400: score += 0.10

        cp = ep["total_complaints"]
        if cp < 60:    score += 0.20
        elif cp < 90:  score += 0.10

        v = ep["total_violations"]
        if v == 0:     score += 0.20
        elif v < 15:   score += 0.10

        f = ep["avg_fairness"]
        if f > 0.7:    score += 0.20
        elif f > 0.5:  score += 0.10

        return round(min(score, 1.0), 4)

    def _score_hard(self, ep: dict) -> float:
        score = 0.0

        ds = ep["demand_satisfaction"]
        if ds > 0.85:   score += 0.20
        elif ds > 0.70: score += 0.10

        v = ep["total_violations"]
        if v < 15:     score += 0.20
        elif v < 30:   score += 0.10

        c = ep["total_cost"]
        if c < 1500:   score += 0.15
        elif c < 2000: score += 0.08

        cp = ep["total_complaints"]
        if cp < 50:    score += 0.15
        elif cp < 80:  score += 0.08

        st = ep["system_trust"]
        if st > 0.8:   score += 0.15
        elif st > 0.6: score += 0.08

        if not ep["collapsed"]:
            score += 0.15

        return round(min(score, 1.0), 4)

    def _vec_to_obs(self, vec: np.ndarray, info: dict = None) -> Observation:
        """
        Convert numpy vector to typed Observation.
        FIX: handles 15-feature vector (index 14 = violations_this_step).
        FIX: battery_level read from vec[10] and included in model.
        FIX: system_trust / battery_level also cross-checked with info dict.
        FIX: demand_supply_ratio denormalized by * 3.0 (matches to_vector).
        """
        n = len(vec)
        info = info or {}

        # Prefer info dict for trust/battery (authoritative from env)
        system_trust   = info.get("system_trust",   float(vec[7])  if n > 7  else 1.0)
        battery_level  = info.get("battery_level",  float(vec[10]) if n > 10 else 0.5)

        # violations: from vec[14] if available, else from info
        if n > 14:
            violations_this_step = int(float(vec[14]) * 20)
        else:
            violations_this_step = info.get("violations", 0)

        return Observation(
            power_usage          = float(vec[0]) * 20.0,
            avg_temperature      = float(vec[1]) * 40.0,
            avg_occupancy        = float(vec[2]),
            complaint_level      = int(float(vec[3]) * 20),
            time_of_day          = int(float(vec[4]) * 23),
            carbon_rate          = float(vec[5]),
            current_cost         = float(vec[6]) * 1000.0,
            system_trust         = system_trust,
            peak_hour            = bool(vec[8] > 0.5) if n > 8  else False,
            solar_output         = float(vec[9])       if n > 9  else 0.0,
            battery_level        = battery_level,                          # FIX
            fairness_score       = float(vec[13])      if n > 13 else 1.0,
            demand_supply_ratio  = float(vec[12]) * 3.0 if n > 12 else 1.0,  # FIX: denorm once
            violations_this_step = violations_this_step,                   # FIX
        )

In [ ]:
%%writefile /content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus/app.py
# paste the full final app.py here
# app.py — EnergyMind Dashboard for HF Spaces

import random
from fastapi import FastAPI
from fastapi.responses import JSONResponse, HTMLResponse
from env.openenv_api import HostelGridOpenEnv, Action

app = FastAPI(
    title="EnergyMind",
    description="Human-Aware Energy Optimization Environment",
    version="2.0.0"
)

environments = {}

DASHBOARD_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>EnergyMind — Human-Aware Energy Optimization</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;800&display=swap" rel="stylesheet"/>
<style>
  :root {
    --bg: #060a0f;
    --surface: #0d1520;
    --surface2: #111d2e;
    --border: #1a2d45;
    --green: #00ff88;
    --green-dim: #00cc6a;
    --yellow: #ffd93d;
    --red: #ff4757;
    --blue: #38bdf8;
    --text: #e2eaf4;
    --muted: #5a7a9a;
    --font-display: 'Syne', sans-serif;
    --font-mono: 'Space Mono', monospace;
  }

  * { margin: 0; padding: 0; box-sizing: border-box; }

  body {
    background: var(--bg);
    color: var(--text);
    font-family: var(--font-mono);
    min-height: 100vh;
    overflow-x: hidden;
  }

  /* Animated grid background */
  body::before {
    content: '';
    position: fixed;
    inset: 0;
    background-image:
      linear-gradient(rgba(0,255,136,0.03) 1px, transparent 1px),
      linear-gradient(90deg, rgba(0,255,136,0.03) 1px, transparent 1px);
    background-size: 40px 40px;
    pointer-events: none;
    z-index: 0;
  }

  .container { max-width: 1200px; margin: 0 auto; padding: 0 24px; position: relative; z-index: 1; }

  /* ── HEADER ── */
  header {
    padding: 40px 0 32px;
    border-bottom: 1px solid var(--border);
    margin-bottom: 40px;
  }

  .header-inner {
    display: flex;
    align-items: flex-start;
    justify-content: space-between;
    flex-wrap: wrap;
    gap: 20px;
  }

  .logo-block {}

  .logo {
    font-family: var(--font-display);
    font-size: 2.8rem;
    font-weight: 800;
    letter-spacing: -1px;
    line-height: 1;
    background: linear-gradient(135deg, var(--green) 0%, var(--blue) 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
  }

  .logo span {
    font-weight: 400;
    opacity: 0.6;
  }

  .tagline {
    margin-top: 8px;
    color: var(--muted);
    font-size: 0.8rem;
    letter-spacing: 2px;
    text-transform: uppercase;
  }

  .badges {
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    align-items: center;
    margin-top: 16px;
  }

  .badge {
    padding: 4px 12px;
    border-radius: 2px;
    font-size: 0.7rem;
    font-weight: 700;
    letter-spacing: 1px;
    text-transform: uppercase;
    border: 1px solid;
  }

  .badge-green { color: var(--green); border-color: var(--green); background: rgba(0,255,136,0.07); }
  .badge-blue  { color: var(--blue);  border-color: var(--blue);  background: rgba(56,189,248,0.07); }
  .badge-yellow{ color: var(--yellow);border-color: var(--yellow);background: rgba(255,217,61,0.07); }

  .live-indicator {
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 8px 16px;
    border: 1px solid var(--border);
    border-radius: 2px;
    font-size: 0.75rem;
    color: var(--muted);
    background: var(--surface);
    height: fit-content;
  }

  .live-dot {
    width: 8px; height: 8px;
    border-radius: 50%;
    background: var(--green);
    animation: pulse 2s infinite;
  }

  @keyframes pulse {
    0%, 100% { opacity: 1; transform: scale(1); }
    50% { opacity: 0.4; transform: scale(0.8); }
  }

  /* ── SCORE CARDS ── */
  .score-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 16px;
    margin-bottom: 40px;
  }

  .score-card {
    background: var(--surface);
    border: 1px solid var(--border);
    padding: 24px;
    position: relative;
    overflow: hidden;
    cursor: pointer;
    transition: border-color 0.2s, transform 0.2s;
  }

  .score-card:hover { transform: translateY(-2px); }
  .score-card.easy:hover  { border-color: var(--green); }
  .score-card.medium:hover{ border-color: var(--yellow); }
  .score-card.hard:hover  { border-color: var(--red); }

  .score-card::before {
    content: '';
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 2px;
  }
  .score-card.easy::before   { background: var(--green); }
  .score-card.medium::before { background: var(--yellow); }
  .score-card.hard::before   { background: var(--red); }

  .card-label {
    font-size: 0.65rem;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 12px;
  }

  .card-title {
    font-family: var(--font-display);
    font-size: 1rem;
    font-weight: 600;
    margin-bottom: 20px;
    line-height: 1.3;
  }

  .score-display {
    display: flex;
    align-items: flex-end;
    gap: 8px;
    margin-bottom: 16px;
  }

  .score-number {
    font-family: var(--font-display);
    font-size: 3rem;
    font-weight: 800;
    line-height: 1;
  }
  .easy .score-number   { color: var(--green); }
  .medium .score-number { color: var(--yellow); }
  .hard .score-number   { color: var(--red); }

  .score-denom { color: var(--muted); font-size: 1.2rem; margin-bottom: 6px; }

  .score-bar {
    height: 3px;
    background: var(--border);
    border-radius: 1px;
    overflow: hidden;
  }

  .score-fill {
    height: 100%;
    border-radius: 1px;
    transition: width 1.5s cubic-bezier(0.23, 1, 0.32, 1);
  }
  .easy .score-fill   { background: var(--green); }
  .medium .score-fill { background: var(--yellow); }
  .hard .score-fill   { background: var(--red); }

  /* ── MAIN GRID ── */
  .main-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
    margin-bottom: 40px;
  }

  .panel {
    background: var(--surface);
    border: 1px solid var(--border);
    padding: 24px;
  }

  .panel-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 20px;
    padding-bottom: 16px;
    border-bottom: 1px solid var(--border);
  }

  .panel-title {
    font-family: var(--font-display);
    font-size: 0.9rem;
    font-weight: 600;
    letter-spacing: 1px;
    text-transform: uppercase;
  }

  .panel-tag {
    font-size: 0.65rem;
    color: var(--muted);
    letter-spacing: 1px;
  }

  /* ── REWARD CHART ── */
  .chart-wrap { position: relative; height: 160px; }

  canvas { width: 100% !important; height: 100% !important; }

  /* ── METRICS ── */
  .metrics-list { display: flex; flex-direction: column; gap: 14px; }

  .metric-row {
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 12px;
  }

  .metric-label { font-size: 0.75rem; color: var(--muted); min-width: 120px; }

  .metric-bar-wrap { flex: 1; height: 4px; background: var(--border); border-radius: 2px; overflow: hidden; }

  .metric-bar { height: 100%; border-radius: 2px; transition: width 0.8s ease; }

  .metric-value { font-size: 0.8rem; font-weight: 700; min-width: 50px; text-align: right; }

  /* ── TASK EXPLORER ── */
  .tasks-grid { display: grid; grid-template-columns: repeat(3,1fr); gap: 12px; }

  .task-tile {
    border: 1px solid var(--border);
    padding: 16px;
    cursor: pointer;
    transition: all 0.2s;
    position: relative;
  }

  .task-tile:hover { background: var(--surface2); }
  .task-tile.active-tile { border-color: var(--green); background: rgba(0,255,136,0.05); }

  .task-dot {
    width: 10px; height: 10px;
    border-radius: 50%;
    margin-bottom: 10px;
  }
  .dot-easy   { background: var(--green); box-shadow: 0 0 8px var(--green); }
  .dot-medium { background: var(--yellow); box-shadow: 0 0 8px var(--yellow); }
  .dot-hard   { background: var(--red); box-shadow: 0 0 8px var(--red); }

  .task-tile-name { font-family: var(--font-display); font-size: 0.85rem; font-weight: 600; margin-bottom: 6px; }
  .task-tile-desc { font-size: 0.7rem; color: var(--muted); line-height: 1.5; }

  /* ── ACTION SIMULATOR ── */
  .action-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }

  .action-btn {
    padding: 10px 12px;
    background: var(--surface2);
    border: 1px solid var(--border);
    color: var(--text);
    font-family: var(--font-mono);
    font-size: 0.72rem;
    cursor: pointer;
    text-align: left;
    transition: all 0.15s;
    display: flex;
    align-items: center;
    gap: 8px;
  }

  .action-btn:hover { border-color: var(--green); color: var(--green); }
  .action-btn:active { transform: scale(0.98); }

  .action-id {
    font-size: 0.65rem;
    color: var(--muted);
    border: 1px solid var(--border);
    padding: 1px 5px;
    min-width: 20px;
    text-align: center;
  }

  /* ── LIVE FEED ── */
  .feed-wrap {
    height: 160px;
    overflow-y: auto;
    font-size: 0.72rem;
    line-height: 1.8;
    color: var(--muted);
  }

  .feed-wrap::-webkit-scrollbar { width: 3px; }
  .feed-wrap::-webkit-scrollbar-thumb { background: var(--border); }

  .feed-line { padding: 2px 0; border-bottom: 1px solid rgba(255,255,255,0.03); }
  .feed-line.good { color: var(--green); }
  .feed-line.warn { color: var(--yellow); }
  .feed-line.bad  { color: var(--red); }

  /* ── REWARD OBJECTIVE BREAKDOWN ── */
  .objectives { display: flex; flex-direction: column; gap: 10px; }

  .obj-row { display: flex; align-items: center; gap: 12px; }
  .obj-name { font-size: 0.72rem; color: var(--muted); min-width: 80px; }
  .obj-weight { font-size: 0.65rem; color: var(--muted); min-width: 30px; }

  .obj-bar-wrap { flex: 1; height: 6px; background: var(--border); border-radius: 3px; overflow: hidden; }
  .obj-bar { height: 100%; border-radius: 3px; }

  .obj-value { font-size: 0.75rem; font-weight: 700; min-width: 45px; text-align: right; }

  /* ── FULL WIDTH PANELS ── */
  .full-panel {
    background: var(--surface);
    border: 1px solid var(--border);
    padding: 24px;
    margin-bottom: 20px;
  }

  /* ── ARCHITECTURE ── */
  .arch-flow {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 0;
    flex-wrap: wrap;
    padding: 20px 0;
  }

  .arch-node {
    background: var(--surface2);
    border: 1px solid var(--border);
    padding: 14px 20px;
    text-align: center;
    min-width: 120px;
  }

  .arch-node-title { font-family: var(--font-display); font-size: 0.8rem; font-weight: 600; margin-bottom: 4px; }
  .arch-node-sub { font-size: 0.65rem; color: var(--muted); }

  .arch-arrow {
    color: var(--green);
    font-size: 1.2rem;
    padding: 0 8px;
    opacity: 0.6;
  }

  /* ── FOOTER ── */
  footer {
    border-top: 1px solid var(--border);
    padding: 24px 0;
    margin-top: 40px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    flex-wrap: wrap;
    gap: 12px;
    color: var(--muted);
    font-size: 0.72rem;
  }

  .footer-links { display: flex; gap: 20px; }
  .footer-links a { color: var(--muted); text-decoration: none; transition: color 0.2s; }
  .footer-links a:hover { color: var(--green); }

  /* ── ANIMATIONS ── */
  .fade-in { animation: fadeIn 0.6s ease forwards; opacity: 0; }
  @keyframes fadeIn { to { opacity: 1; } }

  .slide-up { animation: slideUp 0.5s ease forwards; opacity: 0; transform: translateY(20px); }
  @keyframes slideUp { to { opacity: 1; transform: translateY(0); } }

  /* delays */
  .d1 { animation-delay: 0.1s; }
  .d2 { animation-delay: 0.2s; }
  .d3 { animation-delay: 0.3s; }
  .d4 { animation-delay: 0.4s; }
  .d5 { animation-delay: 0.5s; }

  @media (max-width: 768px) {
    .score-grid { grid-template-columns: 1fr; }
    .main-grid  { grid-template-columns: 1fr; }
    .tasks-grid { grid-template-columns: 1fr; }
    .logo { font-size: 2rem; }
  }
</style>
</head>
<body>

<div class="container">

  <!-- HEADER -->
  <header class="fade-in">
    <div class="header-inner">
      <div class="logo-block">
        <div class="logo">EnergyMind<span>++</span></div>
        <div class="tagline">Human-Aware Energy Optimization · RL Environment</div>
        <div class="badges" style="margin-top:14px">
          <span class="badge badge-green">Reinforcement Learning</span>
          <span class="badge badge-blue">Multi-Objective</span>
          <span class="badge badge-yellow">OpenEnv</span>
          <span class="badge badge-green">PyTorch Hackathon</span>
        </div>
      </div>
      <div class="live-indicator">
        <div class="live-dot"></div>
        SYSTEM ONLINE
      </div>
    </div>
  </header>

  <!-- SCORE CARDS -->
  <div class="score-grid">
    <div class="score-card easy slide-up d1">
      <div class="card-label">Task 1 · Easy</div>
      <div class="card-title">Commitment-Aware<br/>Energy Allocation</div>
      <div class="score-display">
        <div class="score-number" id="score-easy">0.65</div>
        <div class="score-denom">/ 1.00</div>
      </div>
      <div class="score-bar"><div class="score-fill" id="bar-easy" style="width:0%"></div></div>
    </div>
    <div class="score-card medium slide-up d2">
      <div class="card-label">Task 2 · Medium</div>
      <div class="card-title">Fair Enforcement<br/>Under Misuse</div>
      <div class="score-display">
        <div class="score-number" id="score-medium">0.90</div>
        <div class="score-denom">/ 1.00</div>
      </div>
      <div class="score-bar"><div class="score-fill" id="bar-medium" style="width:0%"></div></div>
    </div>
    <div class="score-card hard slide-up d3">
      <div class="card-label">Task 3 · Hard</div>
      <div class="card-title">Crisis Governance<br/>Under Extreme Conditions</div>
      <div class="score-display">
        <div class="score-number" id="score-hard">1.00</div>
        <div class="score-denom">/ 1.00</div>
      </div>
      <div class="score-bar"><div class="score-fill" id="bar-hard" style="width:0%"></div></div>
    </div>
  </div>

  <!-- MAIN GRID -->
  <div class="main-grid">

    <!-- Reward Chart -->
    <div class="panel slide-up d2">
      <div class="panel-header">
        <div class="panel-title">Reward Curve</div>
        <div class="panel-tag">TRAINING PROGRESSION · TASK 3</div>
      </div>
      <div class="chart-wrap">
        <canvas id="rewardChart"></canvas>
      </div>
    </div>

    <!-- Multi-Objective Breakdown -->
    <div class="panel slide-up d3">
      <div class="panel-header">
        <div class="panel-title">Reward Objectives</div>
        <div class="panel-tag">WEIGHT · CONTRIBUTION</div>
      </div>
      <div class="objectives">
        <div class="obj-row">
          <div class="obj-name">⚡ Energy</div>
          <div class="obj-weight">35%</div>
          <div class="obj-bar-wrap"><div class="obj-bar" id="obj-energy" style="width:0%;background:#00ff88"></div></div>
          <div class="obj-value" style="color:#00ff88">+0.35</div>
        </div>
        <div class="obj-row">
          <div class="obj-name">😊 Comfort</div>
          <div class="obj-weight">30%</div>
          <div class="obj-bar-wrap"><div class="obj-bar" id="obj-comfort" style="width:0%;background:#38bdf8"></div></div>
          <div class="obj-value" style="color:#38bdf8">+0.30</div>
        </div>
        <div class="obj-row">
          <div class="obj-name">🌱 Carbon</div>
          <div class="obj-weight">20%</div>
          <div class="obj-bar-wrap"><div class="obj-bar" id="obj-carbon" style="width:0%;background:#ffd93d"></div></div>
          <div class="obj-value" style="color:#ffd93d">+0.20</div>
        </div>
        <div class="obj-row">
          <div class="obj-name">⚖️ Fairness</div>
          <div class="obj-weight">15%</div>
          <div class="obj-bar-wrap"><div class="obj-bar" id="obj-fairness" style="width:0%;background:#ff4757"></div></div>
          <div class="obj-value" style="color:#ff4757">+0.15</div>
        </div>
      </div>

      <div style="margin-top:20px;padding-top:16px;border-top:1px solid var(--border)">
        <div style="font-size:0.65rem;color:var(--muted);margin-bottom:10px;letter-spacing:1px">LIVE METRICS</div>
        <div class="metrics-list">
          <div class="metric-row">
            <div class="metric-label">Demand Satisfaction</div>
            <div class="metric-bar-wrap"><div class="metric-bar" id="m-demand" style="width:0%;background:var(--green)"></div></div>
            <div class="metric-value" style="color:var(--green)" id="mv-demand">93%</div>
          </div>
          <div class="metric-row">
            <div class="metric-label">System Trust</div>
            <div class="metric-bar-wrap"><div class="metric-bar" id="m-trust" style="width:0%;background:var(--blue)"></div></div>
            <div class="metric-value" style="color:var(--blue)" id="mv-trust">87%</div>
          </div>
          <div class="metric-row">
            <div class="metric-label">Fairness Score</div>
            <div class="metric-bar-wrap"><div class="metric-bar" id="m-fair" style="width:0%;background:var(--yellow)"></div></div>
            <div class="metric-value" style="color:var(--yellow)" id="mv-fair">81%</div>
          </div>
        </div>
      </div>
    </div>

    <!-- Task Explorer -->
    <div class="panel slide-up d3">
      <div class="panel-header">
        <div class="panel-title">Task Explorer</div>
        <div class="panel-tag">3 DIFFICULTY LEVELS</div>
      </div>
      <div class="tasks-grid">
        <div class="task-tile active-tile" onclick="selectTask('easy',this)">
          <div class="task-dot dot-easy"></div>
          <div class="task-tile-name">Easy</div>
          <div class="task-tile-desc">Commitment-aware allocation. Honor approved requests.</div>
        </div>
        <div class="task-tile" onclick="selectTask('medium',this)">
          <div class="task-dot dot-medium"></div>
          <div class="task-tile-name">Medium</div>
          <div class="task-tile-desc">Detect misuse, enforce fairness, cap power.</div>
        </div>
        <div class="task-tile" onclick="selectTask('hard',this)">
          <div class="task-dot dot-hard"></div>
          <div class="task-tile-name">Hard</div>
          <div class="task-tile-desc">Heatwave + exam week + outage + misuse simultaneously.</div>
        </div>
      </div>

      <div style="margin-top:16px;padding:14px;background:var(--surface2);border:1px solid var(--border);font-size:0.72rem;color:var(--muted);line-height:1.7" id="task-detail">
        <span style="color:var(--green)">●</span> TASK 1 — 40% rooms have approved requests with minimum supply guarantees.
        Agent must <span style="color:var(--text)">never violate a commitment</span> — heavy penalty if it does.
        Predictable demand, no extreme events. <span style="color:var(--green)">Score: 0.65/1.00</span>
      </div>
    </div>

    <!-- Action Simulator -->
    <div class="panel slide-up d4">
      <div class="panel-header">
        <div class="panel-title">Action Space</div>
        <div class="panel-tag">6 DISCRETE ACTIONS</div>
      </div>
      <div class="action-grid">
        <button class="action-btn" onclick="logAction(0,'increase_ac','comfort ↑ cost ↑')">
          <span class="action-id">0</span> increase_ac
        </button>
        <button class="action-btn" onclick="logAction(1,'decrease_ac','energy saved')">
          <span class="action-id">1</span> decrease_ac
        </button>
        <button class="action-btn" onclick="logAction(2,'lights_off_empty','silent save')">
          <span class="action-id">2</span> lights_off_empty
        </button>
        <button class="action-btn" onclick="logAction(3,'lights_on','restore light')">
          <span class="action-id">3</span> lights_on
        </button>
        <button class="action-btn" onclick="logAction(4,'defer_heavy_load','shift to off-peak')">
          <span class="action-id">4</span> defer_heavy_load
        </button>
        <button class="action-btn" onclick="logAction(5,'do_nothing','hold state')">
          <span class="action-id">5</span> do_nothing
        </button>
      </div>
      <div style="margin-top:14px">
        <div style="font-size:0.65rem;color:var(--muted);letter-spacing:1px;margin-bottom:8px">ACTION LOG</div>
        <div class="feed-wrap" id="feed"></div>
      </div>
    </div>

  </div>

  <!-- ARCHITECTURE -->
  <div class="full-panel slide-up d4">
    <div class="panel-header">
      <div class="panel-title">Agent Architecture</div>
      <div class="panel-tag">Q-LEARNING WITH EXPERIENCE REPLAY</div>
    </div>
    <div class="arch-flow">
      <div class="arch-node">
        <div class="arch-node-title">🏨 Hostel Env</div>
        <div class="arch-node-sub">20 rooms · students · grid</div>
      </div>
      <div class="arch-arrow">→</div>
      <div class="arch-node" style="border-color:var(--green)">
        <div class="arch-node-title" style="color:var(--green)">Observation</div>
        <div class="arch-node-sub">7–17 dim state vector</div>
      </div>
      <div class="arch-arrow">→</div>
      <div class="arch-node">
        <div class="arch-node-title">🧠 Q-Agent</div>
        <div class="arch-node-sub">ε-greedy · replay buffer</div>
      </div>
      <div class="arch-arrow">→</div>
      <div class="arch-node" style="border-color:var(--blue)">
        <div class="arch-node-title" style="color:var(--blue)">Action</div>
        <div class="arch-node-sub">6 discrete actions</div>
      </div>
      <div class="arch-arrow">→</div>
      <div class="arch-node">
        <div class="arch-node-title">🎯 Reward</div>
        <div class="arch-node-sub">4-objective function</div>
      </div>
      <div class="arch-arrow">↩</div>
    </div>

    <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-top:16px">
      <div style="padding:14px;background:var(--surface2);border:1px solid var(--border);text-align:center">
        <div style="font-family:var(--font-display);font-size:1.4rem;font-weight:800;color:var(--green)">15K</div>
        <div style="font-size:0.65rem;color:var(--muted);margin-top:4px">Replay Buffer</div>
      </div>
      <div style="padding:14px;background:var(--surface2);border:1px solid var(--border);text-align:center">
        <div style="font-family:var(--font-display);font-size:1.4rem;font-weight:800;color:var(--blue)">32</div>
        <div style="font-size:0.65rem;color:var(--muted);margin-top:4px">Batch Size</div>
      </div>
      <div style="padding:14px;background:var(--surface2);border:1px solid var(--border);text-align:center">
        <div style="font-family:var(--font-display);font-size:1.4rem;font-weight:800;color:var(--yellow)">0.98</div>
        <div style="font-size:0.65rem;color:var(--muted);margin-top:4px">Discount γ</div>
      </div>
      <div style="padding:14px;background:var(--surface2);border:1px solid var(--border);text-align:center">
        <div style="font-family:var(--font-display);font-size:1.4rem;font-weight:800;color:var(--red)">1500</div>
        <div style="font-size:0.65rem;color:var(--muted);margin-top:4px">Episodes</div>
      </div>
    </div>
  </div>

  <!-- FOOTER -->
  <footer class="fade-in d5">
    <div>Built for <span style="color:var(--green)">Meta PyTorch × Scaler Hackathon</span> · by Team Raptor </div>
    <div class="footer-links">
      <a href="/docs">API Docs</a>
      <a href="/tasks">Tasks</a>
      <a href="/scores">Scores</a>
      <a href="https://github.com/anshu-ai-arch/hostalgrid-plus-plus" target="_blank">GitHub</a>
    </div>
  </footer>

</div>

<script>
// ── Animate score bars on load ─────────────────────────────
window.addEventListener('load', () => {
  setTimeout(() => {
    document.getElementById('bar-easy').style.width   = '65%';
    document.getElementById('bar-medium').style.width = '90%';
    document.getElementById('bar-hard').style.width   = '100%';

    document.getElementById('obj-energy').style.width  = '88%';
    document.getElementById('obj-comfort').style.width = '75%';
    document.getElementById('obj-carbon').style.width  = '50%';
    document.getElementById('obj-fairness').style.width= '38%';

    document.getElementById('m-demand').style.width = '93%';
    document.getElementById('m-trust').style.width  = '87%';
    document.getElementById('m-fair').style.width   = '81%';
  }, 400);
});

// ── Reward Chart ───────────────────────────────────────────
const ctx = document.getElementById('rewardChart').getContext('2d');
const episodes = [100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500];
const rewards  = [204,205,206,204,207,205,208,203,201,201,210,215,221,215,209];

// Draw manually with canvas
function drawChart() {
  const canvas = document.getElementById('rewardChart');
  const c = canvas.getContext('2d');
  const W = canvas.offsetWidth; const H = canvas.offsetHeight;
  canvas.width = W; canvas.height = H;

  const minR = 195, maxR = 225;
  const pad = { top:10, right:10, bottom:30, left:40 };
  const cw = W - pad.left - pad.right;
  const ch = H - pad.top - pad.bottom;

  c.clearRect(0,0,W,H);

  // Grid lines
  c.strokeStyle = 'rgba(26,45,69,0.8)';
  c.lineWidth = 1;
  for(let i=0;i<=4;i++){
    const y = pad.top + (ch/4)*i;
    c.beginPath(); c.moveTo(pad.left,y); c.lineTo(W-pad.right,y); c.stroke();
    c.fillStyle = '#5a7a9a';
    c.font = '9px Space Mono,monospace';
    c.fillText(Math.round(maxR - (maxR-minR)/4*i), 0, y+3);
  }

  // X labels
  c.fillStyle = '#5a7a9a';
  c.font = '9px Space Mono,monospace';
  [100,500,1000,1500].forEach(ep => {
    const idx = episodes.indexOf(ep);
    if(idx<0) return;
    const x = pad.left + (idx/(episodes.length-1))*cw;
    c.fillText(ep, x-8, H-8);
  });

  // Area fill
  const grad = c.createLinearGradient(0, pad.top, 0, H-pad.bottom);
  grad.addColorStop(0, 'rgba(0,255,136,0.15)');
  grad.addColorStop(1, 'rgba(0,255,136,0)');
  c.fillStyle = grad;
  c.beginPath();
  episodes.forEach((ep,i) => {
    const x = pad.left + (i/(episodes.length-1))*cw;
    const y = pad.top + ch - ((rewards[i]-minR)/(maxR-minR))*ch;
    if(i===0) c.moveTo(x,y); else c.lineTo(x,y);
  });
  c.lineTo(pad.left+cw, H-pad.bottom);
  c.lineTo(pad.left, H-pad.bottom);
  c.closePath(); c.fill();

  // Line
  c.strokeStyle = '#00ff88';
  c.lineWidth = 2;
  c.lineJoin = 'round';
  c.beginPath();
  episodes.forEach((ep,i) => {
    const x = pad.left + (i/(episodes.length-1))*cw;
    const y = pad.top + ch - ((rewards[i]-minR)/(maxR-minR))*ch;
    if(i===0) c.moveTo(x,y); else c.lineTo(x,y);
  });
  c.stroke();

  // Dots
  episodes.forEach((ep,i) => {
    const x = pad.left + (i/(episodes.length-1))*cw;
    const y = pad.top + ch - ((rewards[i]-minR)/(maxR-minR))*ch;
    c.beginPath();
    c.arc(x,y,3,0,Math.PI*2);
    c.fillStyle = '#00ff88';
    c.fill();
  });

  // Highlight peak
  const peakIdx = 12; // ep 1300 = 221
  const px = pad.left + (peakIdx/(episodes.length-1))*cw;
  const py = pad.top + ch - ((rewards[peakIdx]-minR)/(maxR-minR))*ch;
  c.beginPath(); c.arc(px,py,5,0,Math.PI*2);
  c.fillStyle = '#fff'; c.fill();
  c.strokeStyle = '#00ff88'; c.lineWidth=2; c.stroke();
}

drawChart();
window.addEventListener('resize', drawChart);

// ── Task selector ─────────────────────────────────────────
const taskDetails = {
  easy: `<span style="color:var(--green)">●</span> TASK 1 — 40% rooms have approved requests with minimum supply guarantees. Agent must <span style="color:var(--text)">never violate a commitment</span> — heavy penalty if it does. Predictable demand, no extreme events. <span style="color:var(--green)">Score: 0.65/1.00</span>`,
  medium: `<span style="color:var(--yellow)">●</span> TASK 2 — 15% of students are selfish and randomly spike demand. Agent must detect spikes and apply power caps. Fairness score tracked across all rooms. <span style="color:var(--yellow)">Score: 0.90/1.00</span>`,
  hard: `<span style="color:var(--red)">●</span> TASK 3 — Combined: 🌡 Heatwave + 📚 Exam week + ⚡ Partial outage + ⚠️ Misuse + 📉 Partial observability. Battery management and solar harvesting critical. <span style="color:var(--red)">Score: 1.00/1.00 ⭐</span>`,
};

function selectTask(id, el) {
  document.querySelectorAll('.task-tile').forEach(t => t.classList.remove('active-tile'));
  el.classList.add('active-tile');
  document.getElementById('task-detail').innerHTML = taskDetails[id];
}

// ── Action feed ───────────────────────────────────────────
const actionMsgs = {
  0: { cls:'warn', msg:'↑ AC increased — comfort up, cost rising' },
  1: { cls:'good', msg:'↓ AC decreased — energy saved' },
  2: { cls:'good', msg:'💡 Lights off in empty rooms — silent save' },
  3: { cls:'',     msg:'💡 Lights restored' },
  4: { cls:'good', msg:'⏱ Heavy load deferred to off-peak — big save' },
  5: { cls:'',     msg:'— Holding current state' },
};

function logAction(id, name, desc) {
  const feed = document.getElementById('feed');
  const now = new Date().toLocaleTimeString('en-US',{hour12:false});
  const { cls, msg } = actionMsgs[id];
  const line = document.createElement('div');
  line.className = `feed-line ${cls}`;
  line.innerHTML = `<span style="color:var(--muted)">${now}</span>  [ACTION ${id}] ${msg}`;
  feed.prepend(line);
  if(feed.children.length > 30) feed.removeChild(feed.lastChild);
}

// Auto-simulate actions
const autoActions = [4,2,0,5,1,4,2,5,0,3];
let autoIdx = 0;
setInterval(() => {
  const id = autoActions[autoIdx % autoActions.length];
  logAction(id, '', '');
  autoIdx++;
}, 3000);

// Init feed
logAction(4,'defer_heavy_load','shift to off-peak');
logAction(2,'lights_off_empty','silent save');
</script>
</body>
</html>"""

@app.get("/", response_class=HTMLResponse)
def root():
    return DASHBOARD_HTML

@app.get("/reset")
@app.post("/reset")
def reset(task_id: str = "task_easy"):
    env = HostelGridOpenEnv(task_id=task_id)
    environments[task_id] = env
    obs = env.reset()
    return JSONResponse({"task_id": task_id, "observation": obs.model_dump(), "state": env.state()})

@app.post("/step")
def step(task_id: str = "task_easy", action_id: int = 5):
    if task_id not in environments:
        env = HostelGridOpenEnv(task_id=task_id)
        environments[task_id] = env
        env.reset()
    env = environments[task_id]
    action = Action(action_id=action_id)
    obs, reward, done, info = env.step(action)
    return JSONResponse({
        "task_id": task_id,
        "observation": obs.model_dump(),
        "reward": reward.value,
        "done": done,
        "info": info,
        "score": env.score() if done else None
    })

@app.get("/state")
def state(task_id: str = "task_easy"):
    if task_id not in environments:
        env = HostelGridOpenEnv(task_id=task_id)
        environments[task_id] = env
        env.reset()
    return JSONResponse(environments[task_id].state())

@app.get("/tasks")
def tasks():
    return JSONResponse({"tasks": [
        {"id": "task_easy",   "name": "Commitment-Aware Energy Allocation",      "difficulty": "easy",   "max_score": 1.0},
        {"id": "task_medium", "name": "Fair Enforcement Under Misuse",            "difficulty": "medium", "max_score": 1.0},
        {"id": "task_hard",   "name": "Crisis Governance Under Extreme Conditions","difficulty": "hard",  "max_score": 1.0},
    ]})

@app.get("/scores")
def scores():
    result = {}
    for task_id in ["task_easy", "task_medium", "task_hard"]:
        env = HostelGridOpenEnv(task_id=task_id)
        env.reset()
        done = False
        while not done:
            _, _, done, _ = env.step(Action(action_id=random.randint(0, 5)))
        result[task_id] = env.score()
    result["average"] = round(sum(result.values()) / len(result), 4)
    return JSONResponse(result)

def main():
    import uvicorn
    uvicorn.run("app:app", host="0.0.0.0", port=7860)

if __name__ == "__main__":
    main()


In [ ]:
REPO = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus"

!git -C "$REPO" checkout ujjwal-feature
!git -C "$REPO" status
!git -C "$REPO" diff --stat


In [ ]:
!git -C "$REPO" commit -m "Add OpenEnv wrapper, dashboard app, and LLM training datasets"


In [ ]:
REPO = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus"

!git -C "$REPO" config user.name "lucifernarayan"
!git -C "$REPO" config user.email "narayanujjwal4192@gmail.com"


In [ ]:
!git -C "$REPO" config user.name
!git -C "$REPO" config user.email


In [ ]:
!git -C "$REPO" commit -m "Add OpenEnv wrapper, dashboard app, and LLM training datasets"


In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
repo_url = f"https://{token}@github.com/anshu-ai-arch/hostalgrid-plus-plus.git"

!git -C "$REPO" push {repo_url} ujjwal-feature


In [ ]:
!git -C "$REPO" status
!git -C "$REPO" log --oneline -n 5


In [ ]:
!git -C "$REPO" add app.py env/openenv_api.py data
!git -C "$REPO" status


In [ ]:
!git -C "$REPO" commit -m "Add OpenEnv wrapper updates, dashboard app, and LLM training datasets"


In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
repo_url = f"https://{token}@github.com/anshu-ai-arch/hostalgrid-plus-plus.git"

!git -C "$REPO" push {repo_url} ujjwal-feature


In [ ]:
!git -C "$REPO" status
!git -C "$REPO" log --oneline -n 3


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO = "/content/drive/MyDrive/hostelgrid-work/hostalgrid-plus-plus"

!find /content/drive/MyDrive -maxdepth 4 -type d -name "hostalgrid-plus-plus"
!git -C "$REPO" status


In [ ]:
!git -C "$REPO" fetch origin


In [ ]:
!git -C "$REPO" checkout main


In [ ]:
!git -C "$REPO" pull origin main


In [ ]:
!git -C "$REPO" checkout -b ujjwal-llm-openenv-final


In [ ]:
!git -C "$REPO" cherry-pick 0ca9ea0


In [ ]:
!git -C "$REPO" branch
!git -C "$REPO" status


In [ ]:
!git -C "$REPO" add app.py env/openenv_api.py data


In [ ]:
!find "$REPO" -maxdepth 2 -type d -name "data"
!find "$REPO" -maxdepth 2 -type f | grep "/data/" | head -20


In [ ]:
!git -C "$REPO" add app.py env/openenv_api.py
!git -C "$REPO" status


In [ ]:
!git -C "$REPO" commit -m "Add OpenEnv wrapper updates and dashboard app"


In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
repo_url = f"https://{token}@github.com/anshu-ai-arch/hostalgrid-plus-plus.git"

!git -C "$REPO" push -u {repo_url} ujjwal-llm-openenv-final


In [ ]:
!find /content/drive/MyDrive -maxdepth 4 -name "*.ipynb"


In [ ]:
!find /content/drive/MyDrive -maxdepth 5 -name "EnergyMind_OpenEnv_TRL_Pipeline.ipynb"


In [ ]:
!find /content/drive/MyDrive -maxdepth 5 -name "*.ipynb" | head -100


In [ ]:
!find /content/drive/MyDrive -maxdepth 5 -iname "*openenv*"
